# 04-loader — Nạp dữ liệu theo phiên

**Câu hỏi**: bước 04 chia dữ liệu thành các phiên như thế nào, phần nào ngẫu nhiên, và phần ngẫu nhiên đó khác nhau bao nhiêu giữa các seed? (D16, D22, Q04.03)

### Nguồn gốc (provenance)

In commit và trạng thái dirty của code, rồi đặt `XLA_FLAGS` trước mọi lần import jax (mẫu ở `lab/README.vi.md` §11).

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit aef4428 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `aef4428`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 04 và so với ảnh chụp

Dùng `replay.rerun` (README §5): dựng trạng thái mới, chạy hàm `s04_loader` (gọi `main.Dataloader` như `main.py`), rồi so từng khoá với ảnh chụp `snap/04.pkl` của run `gc10_v2_ft_r0_s4` (r0, seed 4; QD19, QD23). Quy tắc đạt: dữ liệu và nhãn phải trùng tuyệt đối. Sau đó in shape và số ảnh mỗi lớp của sáu tập mà bước 04 tạo ra.

In [2]:
# cell 1: step 04 again from a fresh state, compared with the snapshot of gc10_v2_ft_r0_s4 (replay.rerun, QD23);
#         then the shape and the per-class counts of the six sets step 04 makes
import json
import numpy as np, pandas as pd
import data_io
RUN = "gc10_v2_ft_r0_s4"
out04, ok = replay.rerun(RUN, "04")
assert ok, "step 04 does not reproduce the snapshot"
names = json.loads(Path(data_io.SPLITS["gc10"]).read_text())["classes"]
SETS = {"s0": ("s0_x", "s0_y"), "pool": ("pool_x", "true_pool_y"), "test0": ("test0_x", "true_test0_y"),
        "test1": ("test1_x", "true_test1_y"), "test_old": ("test_old_x", "true_test_old_y"),
        "test_all": ("test_all_x", "true_test_all_y")}
print({k: out04[x].shape for k, (x, y) in SETS.items()}, "| offset", out04["offset"])
print(pd.DataFrame({k: np.bincount(np.asarray(out04[y]).astype(int), minlength=len(names)) for k, (x, y) in SETS.items()},
                   index=names).to_string())

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 04 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
  OK   s0_x         array float32          max|diff| 0
  OK   s0_y         array float64          max|diff| 0
  OK   pool_x       array float32          max|diff| 0
  OK   test0_x      array float32          max|diff| 0
  OK   test1_x      array float32          max|diff| 0
  OK   test_old_x   array float32          max|diff| 0
  OK   test_all_x   array float32          max|diff| 0
  OK   offset       int                    max|diff| nan
  OK   true_pool_y  array float64          max|diff| 0
  OK   true_test0_y array int64            max|diff| 0
  OK   true_test1_y array int64            max|diff| 0
  OK   true_test_old_y array int64            max|diff| 0
  OK   true_test_all_y array int64            max|diff| 0
rerun step 04: OK (13/13 keys)
{'s0': (764, 768), 'pool': (

- **Kết quả**: `rerun` bước 04 đạt 13/13 khoá, mọi mảng trùng tuyệt đối (max|diff| = 0); driver GPU hiện tại 621.01, lúc chạy run là 617.42. Sáu tập (× 768 chiều), `offset` 5:
  - `s0` 764 = 137 + 170 + 143 + 184 + 130 (5 lớp cũ, đúng `n_use` của file split);
  - `pool` 748 = 10 ảnh mỗi lớp cũ + silk 420, inclusion 135, rolled 17, crease 32, waist 94;
  - `test0` 241 (5 lớp cũ), `test1` 214 (5 lớp mới);
  - `test_old` 241, cùng số ảnh mỗi lớp với `test0`;
  - `test_all` 455 = `test0` + `test1` (tự cộng).
- **Ý nghĩa**: bước 04 tái lập được hoàn toàn từ đặc trưng và seed: chạy lại cho đúng các tập mà run đã dùng. Số ảnh khớp `00-data-gc10` §1.7 (pool 748, test lớp mới 214). Bảng chỉ có số ảnh, chưa cho biết `test_old` có đúng là các ảnh của `test0` hay không; 3.1 kiểm từng hàng.

## 2. Code

### 2.1 Code của bước 04

In nguyên văn bằng `inspect.getsource`, không chép tay:
- `main.load_mode`, nhánh `t1` (gọi `makeT1Loader`);
- `GC10Loader.makeT1Loader`;
- `SplitCountIncrementalLoader._take`: tách mỗi lớp thành `n_use` hàng đầu (theo thứ tự file đặc trưng) và phần còn lại;
- `SplitCountIncrementalLoader._train_data_mix`: ghép phiên S0 (ảnh có nhãn của lớp cũ) và phiên S1 (10 ảnh mỗi lớp cũ bốc từ reserve, cộng ảnh lớp mới), và sửa D22 (nhãn dựng từ đúng các hàng đã lấy).

In [3]:
# cell 2: the code of step 04 as it is (inspect.getsource): main.load_mode (t1 branch), GC10Loader.makeT1Loader,
#         SplitCountIncrementalLoader._take and _train_data_mix
import inspect, main
from dataloaders import gc10
for f in (main.load_mode, gc10.GC10Loader.makeT1Loader, gc10.SplitCountIncrementalLoader._take,
          gc10.SplitCountIncrementalLoader._train_data_mix):
    print(f"# ---- {Path(inspect.getsourcefile(f)).name}: {f.__qualname__}\n{inspect.getsource(f)}")

# ---- main.py: load_mode
def load_mode(args, loader):
    if args.load_mode == 't5':
        return loader.makeT5Loader()
    elif args.load_mode == 'vin':
        return loader.makeVinLoader()
    elif args.load_mode == 't10':
        return loader.makeT10Loader()
    elif args.load_mode == 't1':
        return loader.makeT1Loader()
    else:
        raise ValueError('Load mode not supported')

# ---- gc10.py: GC10Loader.makeT1Loader
    def makeT1Loader(self):
        base, increment, num_known_inc = 5, 5, 10        # 10 old samples/class/session ~ 5% of a base class
        split = json.load(open(SPLIT))
        n_use = [split["n_use"][c] for c in split["classes"]]
        assert (self.args.base, self.args.increment, self.args.num_classes) == (base, increment, 10), \
            "GC10 t1: run with --base 5 --increment 5 --num_classes 10"
        assert self.args.num_dim < min(n_use[:base]) or getattr(self.args, 'cov_prior_kappa', 0) > 0 or getattr(self.args, 'estimator', 'svi') == 

- **Kết quả**:
  - `load_mode` chỉ chuyển `t1` sang `makeT1Loader`.
  - `makeT1Loader` cố định 5 lớp cũ + 5 lớp mới, `num_known_inc` = 10, và có ba phép kiểm: cấu hình 5/5/10; `num_dim` < lớp có nhãn nhỏ nhất (D21); số nhãn trong file đặc trưng khớp file split.
  - `_take` lấy `n_use` hàng đầu của mỗi lớp theo thứ tự file đặc trưng, phần còn lại đưa vào pool.
  - `_train_data_mix`:
    - phiên 0: ảnh có nhãn của lớp cũ thành S0, phần dư của lớp cũ thành pool (reserve);
    - phiên 1: `_sample_known_instance` bốc ảnh cũ từ pool; ghép với ảnh lớp mới (`n_use` hàng đầu); phần dư của lớp mới cũng đưa vào pool.
    
    Nhãn dựng bằng `np.ones(len(use)) * idx`, theo đúng số hàng đã lấy (sửa D22). `_resort_data` được gọi sau mỗi lần ghép.
- **Ý nghĩa**:
  - Ảnh nào là S0, S1 hay reserve được quyết định hoàn toàn bởi thứ tự hàng và `n_use`, không có gì ngẫu nhiên (khớp cột `role` của `data_io`, `00-data-gc10` §1.8).
  - Ngẫu nhiên chỉ nằm trong hai hàm của lớp cha: `_sample_known_instance` và `_resort_data`; 2.2 xem hai hàm này.
  - Phần dư của lớp mới (vai trò `unused`) được đưa vào pool sau phiên 1 nhưng không bao giờ được dùng, vì t1 chỉ có một phiên online.

### 2.2 Phần ngẫu nhiên: bốc ảnh cũ và xáo thứ tự

In nguyên văn `_sample_known_instance` và `_resort_data` của `StrictPerClassIncrementalLoader` (`utils/dataloader.py`).

**Cách làm**: chạy lại loader một lần với seed của run (4), bọc `np.random.permutation` để ghi lại mỗi lần gọi (cỡ mảng bị hoán vị). Rồi kiểm loader chạy theo cách này vẫn ra đúng `pool_x` và `s0_x` như 1.1: việc bọc chỉ ghi lại các lần gọi, không đổi kết quả.

In [4]:
# cell 3: the random part of step 04: source of _sample_known_instance and _resort_data, and every call to the global
#         np.random.permutation during one more loader run from the run's seed (the wrapper only records the calls)
import types
from utils import dataloader as udl
for f in (udl.StrictPerClassIncrementalLoader._sample_known_instance, udl.StrictPerClassIncrementalLoader._resort_data):
    print(f"# ---- {Path(inspect.getsourcefile(f)).name}: {f.__qualname__}\n{inspect.getsource(f)}")
args = types.SimpleNamespace(**replay.snapshots(RUN)["04"]["meta"]["args"])
calls, perm = [], np.random.permutation
np.random.permutation = lambda n: (calls.append(int(n) if np.isscalar(n) else len(n)), perm(n))[1]
try:
    np.random.seed(args.seed)                                    # as chain() and main.py before the loader
    tr = list(main.Dataloader(args)[0])
finally:
    np.random.permutation = perm
print("np.random.permutation calls, in order (array length):", calls)
print("same pool_x and s0_x as cell 1:", np.array_equal(tr[1]._x, out04["pool_x"]), np.array_equal(tr[0]._x, out04["s0_x"]))

# ---- dataloader.py: StrictPerClassIncrementalLoader._sample_known_instance
    def _sample_known_instance(self, x, y, seen_classes, num_known_inc):
        permutation_idx = np.random.permutation(x.shape[0])
        x = x[permutation_idx]
        y = y[permutation_idx]
        remained_x = []
        remained_y = []
        seen_samples_x = []
        seen_samples_y = []
        for c in seen_classes:
            cc_x = x[y == c]
            cc_y = y[y == c]
            seen_samples_x.append(cc_x[:num_known_inc])
            seen_samples_y.append(cc_y[:num_known_inc])
            remained_x.append(cc_x[num_known_inc:])
            remained_y.append(cc_y[num_known_inc:])

        return x, y , np.concatenate(seen_samples_x), np.concatenate(seen_samples_y)

# ---- dataloader.py: StrictPerClassIncrementalLoader._resort_data
    def _resort_data(self, x, y):
        permutation_idx = np.random.permutation(x.shape[0])
        return x[permutation_idx], y[permutation_idx]

np.random.permut

- **Kết quả**: chạy loader thêm một lần từ seed 4 (cùng `pool_x`, `s0_x` như cell 1; True True). Hàm `np.random.permutation` được gọi 5 lần, theo thứ tự với cỡ mảng: 764, 193, 193, 748, 371.
  - 764: xáo S0 (`_resort_data`, phiên 0);
  - 193: xáo pool sau phiên 0, chỉ có reserve của lớp cũ;
  - 193: hoán vị trong `_sample_known_instance`; 10 hàng đầu của mỗi lớp cũ sau hoán vị này là 50 ảnh vào phiên S1;
  - 748: xáo dữ liệu phiên S1 (50 ảnh cũ + 698 ảnh lớp mới);
  - 371: xáo pool sau phiên 1.
- **Ý nghĩa**:
  - Chỉ có lần gọi thứ ba quyết định 50 ảnh cũ nào vào pool S1. Các lần khác chỉ đổi thứ tự hàng.
  - D16 được xác nhận bằng số: `_sample_known_instance` tính `remained_x` nhưng trả về `x` (cả pool đã hoán vị), nên pool sau phiên 1 có 371 = 193 + 178 hàng (tự cộng: reserve 193, chưa bớt 50 ảnh đã bốc, cộng 178 ảnh `unused` của lớp mới). Nếu bỏ ảnh đã bốc thì pool chỉ còn 321.
  - Với t1 (một phiên online), D16 không đổi kết quả. Ở giao thức nhiều phiên, cùng một ảnh cũ có thể bị bốc lại.
  - Mình đã dự đoán pool phiên 0 có 1069 hàng; sai, vì phần dư của lớp mới chỉ vào pool ở phiên 1.
  - [SUY LUẬN] Thứ tự hàng mà bước 04 trả về phụ thuộc seed qua 4 lần xáo. Các bước dùng RNG chung (05, 09) được đặt lại trạng thái RNG ở lối vào (QD21), nên thứ tự này không lan sang chỗ rút RNG của các bước sau.

## 3. Phiên

### 3.1 Các tập test, từng hàng

Pipeline chấm trên bốn tập (`replay.s08_classifier`, `s14_eval`):
- bảng S0 (bước 08): cả bốn dòng All, Old, Novel, S0 đều chấm trên `test0` (241 ảnh lớp cũ);
- bảng S1 (bước 14): All trên `test_all`, Old trên `test_old`, Novel trên `test1`, S0 trên `test0`.

`test_dataloader(mode="old")` ghép các tập test của mọi phiên trước phiên hiện tại, nên ở phiên 1, `test_old` phải đúng bằng `test0`. Kiểm từng hàng (không chỉ số ảnh) hai điều: `test_old` = `test0`, và `test_all` = `test0` nối tiếp `test1`.

In [5]:
# cell 4: the test sets row by row: test_old = test0, test_all = test0 followed by test1 (features and labels)
eq = lambda a, b: bool(np.array_equal(np.asarray(a), np.asarray(b)))
print("test_old = test0:", eq(out04["test_old_x"], out04["test0_x"]), "| labels:", eq(out04["true_test_old_y"], out04["true_test0_y"]))
print("test_all = [test0; test1]:", eq(out04["test_all_x"], np.vstack([out04["test0_x"], out04["test1_x"]])),
      "| labels:", eq(out04["true_test_all_y"], np.concatenate([out04["true_test0_y"], out04["true_test1_y"]])))

test_old = test0: True | labels: True
test_all = [test0; test1]: True | labels: True


- **Kết quả**: `test_old` trùng `test0` ở mọi hàng (cả đặc trưng lẫn nhãn); `test_all` trùng `test0` nối tiếp `test1`. Cả bốn phép so đều True.
- **Ý nghĩa**:
  - Ở phiên S1, dòng Old của bảng S1 và dòng S0 cùng chấm trên đúng 241 ảnh test lớp cũ; khác nhau chỉ ở cách đổi nhãn, vì Old dùng nhãn đã qua `lut` (bước 10, D13).
  - All = 241 ảnh cũ + 214 ảnh mới. Lớp cũ chiếm 53% tập All (241/455, tự chia), nên chỉ số All bị kéo về phía độ chính xác lớp cũ.
  - [SUY LUẬN] Độ chính xác All là trung bình có trọng số theo số ảnh, nên silk_spot (124 ảnh test) nặng gấp khoảng 21 lần rolled_pit (6 ảnh). Đây là một lý do repo dùng balanced accuracy bên cạnh (Q14.02).

### 3.2 Bước 04 qua 15 ảnh chụp

Mỗi FT seed (r0–r2) có file đặc trưng riêng, nên không so các hàng giữa các r được. Thay vào đó, đổi mỗi hàng về số thứ tự của nó trong file đặc trưng của chính run đó (hàng i = ảnh i, `00-data-gc10` §1.6), rồi so theo ảnh.

**Cách làm**: đọc `snap/04.pkl` của 15 run bằng pickle (file chỉ chứa mảng numpy). Kiểm:
1. S0, phần lớp mới của pool, `test0`, `test1` có gồm đúng cùng các ảnh ở cả 15 run không;
2. 50 ảnh cũ được bốc có giống nhau giữa r0, r1, r2 khi cùng seed không. [SUY LUẬN] Dự đoán là có, vì hoán vị chỉ phụ thuộc seed và cỡ mảng (193);
3. các tập bốc được khác nhau bao nhiêu giữa 5 seed, đo bằng chỉ số Jaccard |A ∩ B| / |A ∪ B|.

**Mốc ngẫu nhiên**: bốc 10 trong n_c ảnh reserve của lớp c thì hai seed trùng trung bình 10 · 10 / n_c ảnh (kỳ vọng của phân bố siêu bội (hypergeometric)).

In [6]:
# cell 5: step 04 across the 15 snapshots (pickle: numpy arrays only): each row mapped to its image (its row in the run's own
#         feature file); fixed parts compared by image; the 50 old pool images drawn per seed and their overlap (Jaccard)
import pickle
split = json.loads(Path(data_io.SPLITS["gc10"]).read_text())
snaps = {d.name.split("__")[1][len("gc10_v2_ft_"):]: pickle.loads((d / "snap" / "04.pkl").read_bytes())
         for d in sorted(Path("runs").glob("*__gc10_v2_ft_r[0-2]_s[0-4]"))}
ids = {}
for name, blob in snaps.items():
    a, o = blob["meta"]["args"], blob["out"]
    F = Path(a["data_dir"]) / a["pretrained_model_name"]
    row = {}
    for part, pre in (("train", ""), ("test", "test_")):
        X = np.load(F / f"{pre}features-{F.name}.npy")
        row[part] = {r.tobytes(): i for i, r in enumerate(X)}
        assert len(row[part]) == len(X), f"{name}: two images with byte-identical feature rows in {part}"
    look = lambda part, A: sorted(row[part][r.tobytes()] for r in np.asarray(A))
    y, pool = np.asarray(o["true_pool_y"]).astype(int), np.asarray(o["pool_x"])
    ids[name] = dict(s0=look("train", o["s0_x"]), s1=look("train", pool[y >= 5]), drawn=look("train", pool[y < 5]),
                     test0=look("test", o["test0_x"]), test1=look("test", o["test1_x"]))
print(len(ids), "snapshots")
for k in ("s0", "s1", "test0", "test1"):
    print(f"  {k}: the same {len(ids['r0_s0'][k])} images in all 15 runs: {all(v[k] == ids['r0_s0'][k] for v in ids.values())}")
print("  drawn 50 old images: the same for r0, r1, r2 at each seed:",
      all(ids[f"r0_s{s}"]["drawn"] == ids[f"r{r}_s{s}"]["drawn"] for s in range(5) for r in range(3)))
jac = lambda A, B: len(set(A) & set(B)) / len(set(A) | set(B))
print("Jaccard of the drawn sets between seeds (r0):")
print(pd.DataFrame([[jac(ids[f"r0_s{i}"]["drawn"], ids[f"r0_s{j}"]["drawn"]) for j in range(5)] for i in range(5)],
                   index=[f"s{i}" for i in range(5)], columns=[f"s{j}" for j in range(5)]).round(3).to_string())
n_res = [len(split["train"][c]) - split["n_use"][c] for c in split["classes"][:5]]
ov = sum(100 / n for n in n_res)                                  # expected overlap: sum over classes of 10 * 10 / n_c
print(f"reserve per old class {n_res}; random draws: expected overlap {ov:.1f} of 50, Jaccard {ov / (100 - ov):.3f}")

15 snapshots
  s0: the same 764 images in all 15 runs: True
  s1: the same 698 images in all 15 runs: True
  test0: the same 241 images in all 15 runs: True
  test1: the same 214 images in all 15 runs: True
  drawn 50 old images: the same for r0, r1, r2 at each seed: True
Jaccard of the drawn sets between seeds (r0):
       s0     s1     s2     s3     s4
s0  1.000  0.176  0.220  0.136  0.176
s1  0.176  1.000  0.250  0.163  0.136
s2  0.220  0.250  1.000  0.163  0.149
s3  0.136  0.163  0.163  1.000  0.176
s4  0.176  0.136  0.149  0.176  1.000
reserve per old class [35, 43, 36, 46, 33]; random draws: expected overlap 13.2 of 50, Jaccard 0.152


- **Kết quả**:
  - Ở cả 15 run, S0 gồm cùng 764 ảnh, phần lớp mới của pool cùng 698 ảnh, `test0` cùng 241 ảnh và `test1` cùng 214 ảnh.
  - 50 ảnh cũ được bốc giống hệt nhau giữa r0, r1, r2 ở mỗi seed.
  - Chỉ số Jaccard giữa các tập bốc của hai seed (r0) là 0.136–0.250; trung bình 10 cặp là 0.175 (tự tính). Mốc ngẫu nhiên là 0.152, ứng với trung bình 13.2/50 ảnh trùng (reserve mỗi lớp: 35, 43, 36, 46, 33).
- **Ý nghĩa**:
  - Ở bước 04, seed phase 2 chỉ đổi hai thứ: 50 ảnh cũ nào vào pool, và thứ tự hàng. FT seed chỉ đổi giá trị đặc trưng, không đổi ảnh nào được chọn. Vì vậy so sánh giữa r0, r1, r2 ở cùng seed là ghép cặp đúng nghĩa: cùng ảnh, khác biểu diễn.
  - Các tập bốc trùng nhau ở mức của bốc ngẫu nhiên (0.175 so với 0.152; 10 cặp không độc lập nên không kiểm định).
  - [SUY LUẬN] Mỗi seed đặt vào pool một nhóm ảnh cũ gần như khác hẳn, nên một phần dao động giữa các seed ở bước 09–14 có thể đến từ việc 50 ảnh cũ nào lẫn vào pool. Phần này chưa tách khỏi dao động do k-means; đó là câu hỏi cho bước 09.

### 3.3 Thứ tự hàng của pool có đổi k-means ở bước 09 không

Bước 04 xáo thứ tự hàng của pool theo seed (2.2). k-means của bước 09 khởi tạo bằng k-means++, chọn tâm bằng RNG trên chỉ số hàng, nên cùng một trạng thái RNG mà thứ tự hàng khác thì có thể ra phân hoạch khác.

**Cách làm**: đặt RNG về trạng thái ở lối vào bước 09 của ảnh chụp (RNG chung), chạy `replay.s09_cluster` hai lần:
- (a) pool theo thứ tự gốc. Phải ra đúng `ids` của ảnh chụp 09, để kiểm cách làm;
- (b) pool theo một hoán vị cố định (`RandomState(0)`, không đụng RNG toàn cục).

Đổi `ids` của (b) về thứ tự gốc, rồi so với (a) bằng chỉ số Rand đã hiệu chỉnh (adjusted Rand index, ARI; Hubert & Arabie 1985). ARI = 1 nghĩa là cùng phân hoạch, ARI gần 0 là trùng ngẫu nhiên. Thêm balanced accuracy ghép một–một trên lớp mới (`score09` của `lab/score.py`), dùng nhãn thật của pool chỉ để chấm.

In [7]:
# cell 6: does the row order of the pool (shuffled by step 04) change the k-means of step 09? Same RNG state at the entry
#         of 09 (common random numbers), original order vs a fixed permutation; ids mapped back to the original order
from sklearn.metrics import adjusted_rand_score
from score import score09
blobs = replay.snapshots(RUN)
y_pool = np.asarray(out04["true_pool_y"]).astype(int)            # true labels: scoring only
def km(order):
    np.random.set_state(blobs["09"]["rng_in"])
    o = replay.s09_cluster({"args": args, "pool_x": out04["pool_x"][order], "offset": out04["offset"]})
    ids = np.empty(len(order), int)
    ids[order] = o["ids"]
    return ids, o["inertia"]
ids_a, in_a = km(np.arange(len(y_pool)))
print("(a) original order = snapshot 09:", np.array_equal(ids_a, blobs["09"]["out"]["ids"]))
ids_b, in_b = km(np.random.RandomState(0).permutation(len(y_pool)))
for tag, ids, inertia in (("(a) original", ids_a, in_a), ("(b) permuted", ids_b, in_b)):
    s = score09(y_pool, ids, [c.split("_")[0] for c in names], 5)
    print(f"{tag}: inertia {inertia:.2f} | bal acc novel (Hungarian) {s['bal_acc_hungarian']:.4f} | best rolled {s['best_rolled']:.3f}, "
          f"crease {s['best_crease']:.3f}")
print(f"ARI (a) vs (b): {adjusted_rand_score(ids_a, ids_b):.4f}")

(array([5, 6, 7, 8, 9], dtype=int32), array([ 95, 157, 272, 121, 103]))
(a) original order = snapshot 09: True
(array([5, 6, 7, 8, 9], dtype=int32), array([259,  97, 223,  79,  90]))
(a) original: inertia 82926.20 | bal acc novel (Hungarian) 0.3167 | best rolled 0.000, crease 0.000
(b) permuted: inertia 82757.47 | bal acc novel (Hungarian) 0.5589 | best rolled 0.000, crease 0.750
ARI (a) vs (b): 0.7329


- **Kết quả**:
  - (a) Thứ tự gốc ra đúng `ids` của ảnh chụp 09 (True): cỡ cụm 95, 157, 272, 121, 103; inertia 82926.20; balanced accuracy lớp mới 0.3167; rolled_pit và crease đều không có cụm riêng (best 0.000).
  - (b) Cùng trạng thái RNG, pool theo một hoán vị khác: cỡ cụm 259, 97, 223, 79, 90; inertia 82757.47 (thấp hơn); balanced accuracy 0.5589; crease có cụm riêng với best 0.750.
  - ARI giữa (a) và (b) là 0.7329.
- **Ý nghĩa**:
  - Thứ tự hàng, do các lần xáo của bước 04 quyết định theo seed, là một đầu vào thật của bước 09. Với cùng trạng thái RNG, k-means++ chọn tâm khởi tạo trên chỉ số hàng, nên đổi thứ tự là đổi khởi tạo.
  - `n_init` 10 không đủ để k-means về cùng một nghiệm: (b) tìm được inertia thấp hơn nghiệm mà pipeline dùng ở r0 seed 4. Tức là nghiệm của pipeline không phải tốt nhất mà k-means có thể đạt.
  - Đây mới là một hoán vị trên một ảnh chụp: chỉ cho biết hiệu ứng có thể lớn, chưa cho biết lớn trung bình bao nhiêu. [SUY LUẬN] Một phần dao động giữa các seed ở bước 09–14 (S1_New 49.07–57.48 ở r0) có thể đến từ khởi tạo k-means chứ không từ dữ liệu. Inertia thấp hơn đi cùng balanced accuracy cao hơn ở cặp này; nếu điều đó đúng rộng rãi, tăng `n_init` hoặc chọn nghiệm theo inertia trên nhiều thứ tự có thể là một cải tiến rẻ ở bước 09.
  - Biến thể ở bước 09 chạy tiếp từ ảnh chụp 04, nên giữ nguyên thứ tự hàng; phép so giữa biến thể và mặc định không bị lẫn hiệu ứng này (RNG chung, QD21).

## Summary (EN)

1. **Step 04 reproduces exactly.** Rerunning step 04 of `gc10_v2_ft_r0_s4` gives all 13 output keys bit-identical to the snapshot (§1.1). The sessions are S0 764 labeled old images, a pool of 748 (10 per old class + silk 420, inclusion 135, rolled 17, crease 32, waist 94), test0 241, test1 214; test_old equals test0 and test_all is test0 followed by test1, row by row (§1.1, §3.1).
2. **Only one random choice selects data.** Membership of S0, S1 and the test sets is fixed by row order and `n_use`. The global RNG is drawn 5 times (array sizes 764, 193, 193, 748, 371); only the third draw picks the 50 old pool images, the rest reorder rows. D16 shows in the numbers: the pool after session 1 keeps all 193 reserve rows plus 178 unused novel rows (§2.2).
3. **Seeds change images, FT seeds change features.** Across the 15 snapshots, S0, the novel part of the pool and both test sets are the same images; the 50 drawn old images are identical across r0–r2 at each seed. Between seeds they overlap at chance level (mean Jaccard 0.175 against 0.152 expected) (§3.2).
4. **Row order changes the k-means of step 09.** With the same RNG state at step 09, a permuted pool gives another partition (ARI 0.733) with lower inertia (82757 vs 82926) and higher balanced novel accuracy (0.559 vs 0.317); crease gets its own cluster (best 0.750 vs 0). One permutation on one snapshot (§3.3). [inference] k-means with `n_init` 10 does not reach a stable optimum, and part of the seed-to-seed spread may come from initialisation, not from the data.